# AIS DS2 Preprocessing and Data Preparation — Final Target Construction Pipeline

This notebook prepares the final **DS2** modelling dataset used by the training and evaluation notebook. It is not the EDA notebook. Its purpose is to convert raw AIS messages into leakage-safe model matrices and diagnosable train/validation/test files.

The pipeline uses 10-minute vessel slices, constructs behavior labels from past/future slice context, filters out unlabeled or empty current slices, engineers 48 current/past-only model features, and persists CSV/JSON files for repeatable training.

Key outputs:

- `slice_level_all.csv`: continuous per-vessel 10-minute slice timeline, including empty slices.
- `target_df.csv`: slice-level dataframe with target-construction diagnostics and labels. Many rows remain unlabeled because empty or ambiguous slices are intentionally not forced into a class.
- `model_df.csv`: final labelled, non-empty, model-ready dataframe.
- `X_train.csv`, `y_train.csv`, `X_valid.csv`, `y_valid.csv`, `X_test.csv`, `y_test.csv`.
- `train_df.csv`, `valid_df.csv`, `test_df.csv`: diagnostic versions of the splits, including `MMSI`, `slice_start`, target-construction columns, and model features.
- `feature_cols.json`, `target_map.json`, `split_summary.json`, `preprocessing_update_summary.json`.

Final target classes are:

1. `steady`
2. `stop`
3. `accelerate`
4. `decelerate`
5. `maneuver`

Important target-construction note: `steady` is **not** used as a residual/default label. It is assigned only when there is positive evidence of stable moving behavior. Rows that do not satisfy any class rule remain unlabeled and are excluded from the final model-ready dataframe.

For serious model evaluation, `SPLIT_BY_MMSI = True` should remain enabled. This gives a vessel-held-out split, so validation and test performance are measured on MMSIs not seen during training.

Expected DS2 output summary after running this notebook:

- Raw DS2 rows: **482,846**
- Continuous slice-level rows: **1,016,245**
- Unique MMSIs: **424**
- Empty slice ratio: **0.8914**
- Target dataframe rows: **1,016,245**
- Unlabeled target rows: **915,690**
- Model-ready rows after filtering labelled, non-empty current slices: **91,021**
- Feature count: **48**
- Split mode: **MMSI-held-out**
- Train / validation / test shapes: **60,365 / 16,096 / 14,560**

The large difference between `target_df` and `model_df` is expected: `target_df` keeps the full continuous timeline for diagnostics, while `model_df` keeps only labelled, non-empty, representative current slices for modelling.


Public output directory used by this notebook: `ais_ds2_prepared/`, with `final/` for the natural train/validation/test split and `balanced_train/` for the train-only balanced training matrices used by the final model notebook.


In [1]:
# =========================================================
# Config
# =========================================================

from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd

DATA_PATH = Path("./Data/DS2.csv")
# Public, stable output folder for all prepared DS2 artifacts.
PREP_DIR = Path("./ais_ds2_prepared")
FINAL_DIR = PREP_DIR / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Raw column names
MMSI_COL = "MMSI"
TIME_COL = "BaseDateTime"
SOG_COL = "SOG"
COG_COL = "COG"

# Slice representation
SLICE_MINUTES = 10
MIN_MSGS_FOR_REPR = 1

# Split settings
RANDOM_STATE = 42
VALID_TEST_SIZE = 0.30
TEST_SIZE_WITHIN_TEMP = 0.50

# Keep True for leakage-safe evaluation by unseen vessels.
# Set False only for quick/simple row-level experiments.
SPLIT_BY_MMSI = True

# Feature construction
N_LAGS = 3

print("DATA_PATH:", DATA_PATH)
print("FINAL_DIR:", FINAL_DIR)
print("SPLIT_BY_MMSI:", SPLIT_BY_MMSI)

DATA_PATH: Data/DS2.csv
FINAL_DIR: ais_ds2_prepared/final
SPLIT_BY_MMSI: True


In [2]:

# =========================================================
# Helper functions: circular math, cleaning, slicing
# =========================================================

def circular_mean_deg(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return np.nan

    rad = np.deg2rad(arr)
    sin_mean = np.mean(np.sin(rad))
    cos_mean = np.mean(np.cos(rad))

    if np.isclose(sin_mean, 0.0) and np.isclose(cos_mean, 0.0):
        return np.nan

    return np.rad2deg(np.arctan2(sin_mean, cos_mean)) % 360


def circular_std_deg(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return np.nan

    rad = np.deg2rad(arr)
    sin_mean = np.mean(np.sin(rad))
    cos_mean = np.mean(np.cos(rad))
    r = np.sqrt(sin_mean**2 + cos_mean**2)

    if r <= 0:
        return np.nan

    return np.rad2deg(np.sqrt(-2 * np.log(r)))


def circular_diff_scalar_deg(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return ((a - b + 180) % 360) - 180


def circular_diff_series_deg(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = ((a - b + 180) % 360) - 180
    out[np.isnan(a) | np.isnan(b)] = np.nan
    return out


def clean_ais_df(df):
    """
    Keep this cleaning logic aligned with the final project preprocessing/EDA decisions.
    The most important point: remove COG == 360 because in this dataset it
    behaved like a suspicious/low-information placeholder.
    """
    df = df.copy()

    required_cols = [MMSI_COL, TIME_COL, SOG_COL, COG_COL]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df[MMSI_COL] = pd.to_numeric(df[MMSI_COL], errors="coerce")
    df[SOG_COL] = pd.to_numeric(df[SOG_COL], errors="coerce")
    df[COG_COL] = pd.to_numeric(df[COG_COL], errors="coerce")

    df = df.dropna(subset=[MMSI_COL, TIME_COL]).copy()
    df[MMSI_COL] = df[MMSI_COL].astype("int64")

    # Basic physical validity
    df.loc[df[SOG_COL] < 0, SOG_COL] = np.nan
    df.loc[(df[COG_COL] < 0) | (df[COG_COL] > 360), COG_COL] = np.nan

    # Final notebook decision: remove suspicious COG == 360 rows.
    df = df[df[COG_COL] != 360].copy()

    return df


def build_full_slice_grid_for_vessel(g, slice_freq):
    """
    Build continuous slice grid for one vessel.

    Important:
    In newer pandas versions, the group column MMSI may not exist inside g
    during groupby().apply(). Therefore we use g.name as the MMSI value.
    """
    mmsi = g.name

    full_index = pd.date_range(
        start=g["slice_start"].min(),
        end=g["slice_start"].max(),
        freq=slice_freq
    )

    full = pd.DataFrame({
        MMSI_COL: mmsi,
        "slice_start": full_index
    })

    return full.merge(
        g.drop(columns=[MMSI_COL], errors="ignore"),
        on="slice_start",
        how="left"
    )


def build_slice_level_for_df(raw_df):
    """
    Converts cleaned AIS rows to global-clock-aligned slice rows.
    Safe to call for one MMSI bucket because no computation depends on other vessels.
    """
    ais_slice_df = clean_ais_df(raw_df)

    if ais_slice_df.empty:
        return pd.DataFrame()

    slice_freq = f"{SLICE_MINUTES}min"
    ais_slice_df = ais_slice_df.sort_values([MMSI_COL, TIME_COL]).copy()

    ais_slice_df["slice_start"] = ais_slice_df[TIME_COL].dt.floor(slice_freq)
    ais_slice_df["slice_end"] = ais_slice_df["slice_start"] + pd.Timedelta(minutes=SLICE_MINUTES)

    agg_df = (
        ais_slice_df
        .groupby([MMSI_COL, "slice_start"], as_index=False)
        .agg(
            n_msgs=(TIME_COL, "size"),
            first_ts_in_slice=(TIME_COL, "min"),
            last_ts_in_slice=(TIME_COL, "max"),

            rep_sog=(SOG_COL, "median"),
            sog_mean=(SOG_COL, "mean"),
            sog_std=(SOG_COL, "std"),
            sog_min=(SOG_COL, "min"),
            sog_max=(SOG_COL, "max"),

            rep_cog=(COG_COL, lambda x: circular_mean_deg(x)),
            cog_circular_std=(COG_COL, lambda x: circular_std_deg(x)),
        )
    )

    agg_df["slice_end"] = agg_df["slice_start"] + pd.Timedelta(minutes=SLICE_MINUTES)
    agg_df["coverage_seconds"] = (
        agg_df["last_ts_in_slice"] - agg_df["first_ts_in_slice"]
    ).dt.total_seconds()

    full_grid_df = (
        agg_df
        .groupby(MMSI_COL, group_keys=False)
        .apply(lambda g: build_full_slice_grid_for_vessel(g, slice_freq))
        .reset_index(drop=True)
    )

    full_grid_df["slice_end"] = full_grid_df["slice_start"] + pd.Timedelta(minutes=SLICE_MINUTES)

    slice_level_df = full_grid_df.copy()
    slice_level_df["has_data"] = slice_level_df["n_msgs"].notna()
    slice_level_df["n_msgs"] = slice_level_df["n_msgs"].fillna(0).astype(int)
    slice_level_df["enough_data_for_repr"] = slice_level_df["n_msgs"] >= MIN_MSGS_FOR_REPR

    slice_level_df["cog_valid_for_maneuver"] = (
        slice_level_df["has_data"] &
        slice_level_df["rep_cog"].notna() &
        slice_level_df["rep_sog"].notna() &
        (slice_level_df["rep_sog"] >= 1.0)
    )

    return slice_level_df.sort_values([MMSI_COL, "slice_start"]).reset_index(drop=True)


In [3]:
# =========================================================
# Step 1: Load raw data and build slice-level dataframe in memory
# =========================================================

raw_df = pd.read_csv(DATA_PATH)
print("Raw df:", raw_df.shape)

slice_level_df = build_slice_level_for_df(raw_df)
slice_level_df = slice_level_df.sort_values([MMSI_COL, "slice_start"]).reset_index(drop=True)

# Persist for diagnostics/reuse
slice_level_df.to_csv(FINAL_DIR / "slice_level_all.csv", index=False)

print("slice_level_df:", slice_level_df.shape)
print("Unique MMSIs  :", slice_level_df[MMSI_COL].nunique())
print("Empty ratio   :", round((~slice_level_df["has_data"]).mean(), 4))
print("Saved:", FINAL_DIR / "slice_level_all.csv")

del raw_df
gc.collect()

Raw df: (482846, 17)


/var/folders/4h/4hjx_9ys7sj8t4wqnpxc2t640000gn/T/ipykernel_18589/3951007649.py:35: RuntimeWarning: invalid value encountered in sqrt
  return np.rad2deg(np.sqrt(-2 * np.log(r)))


slice_level_df: (1016245, 17)
Unique MMSIs  : 424
Empty ratio   : 0.8914
Saved: ais_ds2_prepared/final/slice_level_all.csv


0

### Interpretation of slice-level construction

The raw DS2 file contains **482,846** AIS messages. After cleaning and converting the messages into a continuous 10-minute per-vessel timeline, the notebook creates **1,016,245** slice rows for **424** MMSIs.

The empty-slice ratio is high (**0.8914**) because the full per-vessel timeline keeps 10-minute intervals even when no AIS message exists in that interval. This is intentional: empty slices are useful for target-window diagnostics and for avoiding misleading continuity assumptions. Empty current slices are later excluded from the final modelling dataframe.

In [ ]:
# =========================================================
# Step 3: Construct target labels and persist target_df
# Final target-construction rule set
#
# Final target-construction design:
#   `steady` is no longer assigned as the residual/default class.
#   A row becomes steady only when there is explicit positive evidence of
#   stable moving behavior across the past/current/future context.
#
# Rows that are not stop / maneuver / accelerate / decelerate and also do not
# satisfy the explicit steady rule remain unlabeled (NaN) and are excluded from
# the model-ready dataframe in the next step.
# =========================================================

# Window parameters
PAST_SLICES = 2
FUTURE_SLICES = 2

MIN_PAST_NONEMPTY = 1
MIN_FUTURE_NONEMPTY = 1
MIN_COG_VALID_SLICES = 1

STOP_SOG_THRESHOLD = 0.5
STOP_RATIO_THRESHOLD = 0.80

MANEUVER_TURN_THRESHOLD = 45.0
MIN_VALID_COG_SOG = 1.0

# Speed regimes based on past_mean_rep_sog
LOW_SPEED_MAX = 3.0
MID_SPEED_MAX = 10.0

ACCEL_THRESH_LOW = 0.7
ACCEL_THRESH_MID = 1.0
ACCEL_THRESH_HIGH = 1.5

DECEL_THRESH_LOW = 0.7
DECEL_THRESH_MID = 1.0
DECEL_THRESH_HIGH = 1.5

MIN_SPEED_FOR_ACCEL_DECEL = 1.0

# Moderate steady target-definition thresholds.
# These thresholds intentionally define steady as stable movement, not as
# "everything that is not another class".
MODERATE_STEADY_MIN_PAST_SOG = 1.0
MODERATE_STEADY_MIN_FUTURE_SOG = 1.0
MODERATE_STEADY_MAX_ABS_DELTA_MEAN_SOG = 0.50
MODERATE_STEADY_MAX_WINDOW_SOG_STD = 0.50
MODERATE_STEADY_MAX_TURN_CHANGE_DEG = 35.0
MODERATE_STEADY_MAX_FUTURE_STOP_RATIO = 0.20

TARGET_COL = "target_class"


def summarize_window(window_df):
    out = {}
    out["n_total_slices"] = len(window_df)
    out["n_nonempty"] = int(window_df["has_data"].fillna(False).sum())

    nonempty = window_df[window_df["has_data"]].copy()

    if len(nonempty) > 0:
        rep_sog = pd.to_numeric(nonempty["rep_sog"], errors="coerce")
        out["mean_rep_sog"] = rep_sog.mean()
        out["median_rep_sog"] = rep_sog.median()
        out["std_rep_sog"] = rep_sog.std(ddof=0)
        out["min_rep_sog"] = rep_sog.min()
        out["max_rep_sog"] = rep_sog.max()
        out["range_rep_sog"] = rep_sog.max() - rep_sog.min()
    else:
        out["mean_rep_sog"] = np.nan
        out["median_rep_sog"] = np.nan
        out["std_rep_sog"] = np.nan
        out["min_rep_sog"] = np.nan
        out["max_rep_sog"] = np.nan
        out["range_rep_sog"] = np.nan

    stop_mask = nonempty["rep_sog"] <= STOP_SOG_THRESHOLD if len(nonempty) > 0 else pd.Series(dtype=bool)
    out["stop_ratio"] = stop_mask.mean() if len(nonempty) > 0 else np.nan

    cog_valid = nonempty[nonempty["cog_valid_for_maneuver"]].copy()
    out["n_cog_valid"] = len(cog_valid)
    out["mean_rep_cog"] = circular_mean_deg(cog_valid["rep_cog"].values) if len(cog_valid) > 0 else np.nan
    out["std_rep_cog"] = circular_std_deg(cog_valid["rep_cog"].values) if len(cog_valid) > 1 else np.nan

    return out


def get_speed_regime(speed):
    if pd.isna(speed):
        return np.nan
    if speed < LOW_SPEED_MAX:
        return "low"
    elif speed < MID_SPEED_MAX:
        return "mid"
    return "high"


def get_accel_threshold(speed):
    regime = get_speed_regime(speed)
    if regime == "low":
        return ACCEL_THRESH_LOW
    elif regime == "mid":
        return ACCEL_THRESH_MID
    elif regime == "high":
        return ACCEL_THRESH_HIGH
    return np.nan


def get_decel_threshold(speed):
    regime = get_speed_regime(speed)
    if regime == "low":
        return DECEL_THRESH_LOW
    elif regime == "mid":
        return DECEL_THRESH_MID
    elif regime == "high":
        return DECEL_THRESH_HIGH
    return np.nan


def is_moderate_steady(row):
    """Positive definition of steady movement used for target construction."""
    required = [
        "past_mean_rep_sog",
        "future_mean_rep_sog",
        "delta_mean_sog",
        "future_stop_ratio",
    ]
    if any(pd.isna(row[c]) for c in required):
        return False

    if row["past_mean_rep_sog"] < MODERATE_STEADY_MIN_PAST_SOG:
        return False
    if row["future_mean_rep_sog"] < MODERATE_STEADY_MIN_FUTURE_SOG:
        return False
    if row["future_stop_ratio"] > MODERATE_STEADY_MAX_FUTURE_STOP_RATIO:
        return False
    if abs(row["delta_mean_sog"]) > MODERATE_STEADY_MAX_ABS_DELTA_MEAN_SOG:
        return False

    # Speed variability gate. If a window has only one non-empty slice, std is 0
    # from summarize_window(ddof=0), so this does not unfairly reject sparse but
    # otherwise valid windows.
    if pd.notna(row.get("past_std_rep_sog")) and row["past_std_rep_sog"] > MODERATE_STEADY_MAX_WINDOW_SOG_STD:
        return False
    if pd.notna(row.get("future_std_rep_sog")) and row["future_std_rep_sog"] > MODERATE_STEADY_MAX_WINDOW_SOG_STD:
        return False

    # Heading stability gate. If both past/future COG summaries are valid, the
    # turn must be small. If COG is not available/reliable, steady can still be
    # assigned based on speed stability, but maneuver has already had priority.
    if pd.notna(row.get("turn_change_deg_abs")) and row["turn_change_deg_abs"] > MODERATE_STEADY_MAX_TURN_CHANGE_DEG:
        return False

    return True


def assign_target(row):
    # Validity gates
    if row["past_nonempty_count"] < MIN_PAST_NONEMPTY:
        return np.nan
    if row["future_nonempty_count"] < MIN_FUTURE_NONEMPTY:
        return np.nan

    # 1) STOP
    if pd.notna(row["future_stop_ratio"]) and row["future_stop_ratio"] >= STOP_RATIO_THRESHOLD:
        return "stop"

    # 2) MANEUVER
    if (
        row["past_cog_valid_count"] >= MIN_COG_VALID_SLICES and
        row["future_cog_valid_count"] >= MIN_COG_VALID_SLICES and
        pd.notna(row["turn_change_deg_abs"]) and
        row["turn_change_deg_abs"] >= MANEUVER_TURN_THRESHOLD
    ):
        return "maneuver"

    accel_threshold = get_accel_threshold(row["past_mean_rep_sog"])
    decel_threshold = get_decel_threshold(row["past_mean_rep_sog"])

    # 3) ACCELERATE
    if (
        pd.notna(row["past_mean_rep_sog"]) and
        row["past_mean_rep_sog"] >= MIN_SPEED_FOR_ACCEL_DECEL and
        pd.notna(row["delta_mean_sog"]) and
        pd.notna(accel_threshold) and
        row["delta_mean_sog"] >= accel_threshold
    ):
        return "accelerate"

    # 4) DECELERATE
    if (
        pd.notna(row["past_mean_rep_sog"]) and
        row["past_mean_rep_sog"] >= MIN_SPEED_FOR_ACCEL_DECEL and
        pd.notna(row["delta_mean_sog"]) and
        pd.notna(decel_threshold) and
        row["delta_mean_sog"] <= -decel_threshold
    ):
        return "decelerate"

    # 5) MODERATE STEADY: positive evidence only, not residual fallback.
    if is_moderate_steady(row):
        return "steady"

    return np.nan


target_df = slice_level_df.copy()
target_df = target_df.sort_values([MMSI_COL, "slice_start"]).reset_index(drop=True)

# ---------------------------------------------------------
# Diagnostic columns used for label construction
# ---------------------------------------------------------

numeric_target_columns = [
    "past_nonempty_count",
    "future_nonempty_count",
    "past_mean_rep_sog",
    "future_mean_rep_sog",
    "delta_mean_sog",
    "past_std_rep_sog",
    "future_std_rep_sog",
    "past_range_rep_sog",
    "future_range_rep_sog",
    "future_stop_ratio",
    "past_cog_valid_count",
    "future_cog_valid_count",
    "past_mean_rep_cog",
    "future_mean_rep_cog",
    "past_std_rep_cog",
    "future_std_rep_cog",
    "turn_change_deg",
    "turn_change_deg_abs",
    "accel_threshold_used",
    "decel_threshold_used",
]

for col in numeric_target_columns:
    target_df[col] = np.nan

# String/categorical diagnostic column
target_df["speed_regime"] = pd.Series(pd.NA, index=target_df.index, dtype="object")

# ---------------------------------------------------------
# Construct past/future summaries and diagnostic variables
# ---------------------------------------------------------

for _, idx in target_df.groupby(MMSI_COL).groups.items():
    vessel = target_df.loc[idx].sort_values("slice_start")
    positions = vessel.index.to_list()

    for pos_i, row_idx in enumerate(positions):
        past_start = max(0, pos_i - PAST_SLICES)
        past_end = pos_i
        future_start = pos_i + 1
        future_end = min(len(positions), pos_i + 1 + FUTURE_SLICES)

        past_idx = positions[past_start:past_end]
        future_idx = positions[future_start:future_end]

        if len(past_idx) == 0 or len(future_idx) == 0:
            continue

        past_summary = summarize_window(target_df.loc[past_idx])
        future_summary = summarize_window(target_df.loc[future_idx])

        target_df.at[row_idx, "past_nonempty_count"] = past_summary["n_nonempty"]
        target_df.at[row_idx, "future_nonempty_count"] = future_summary["n_nonempty"]

        target_df.at[row_idx, "past_mean_rep_sog"] = past_summary["mean_rep_sog"]
        target_df.at[row_idx, "future_mean_rep_sog"] = future_summary["mean_rep_sog"]
        target_df.at[row_idx, "past_std_rep_sog"] = past_summary["std_rep_sog"]
        target_df.at[row_idx, "future_std_rep_sog"] = future_summary["std_rep_sog"]
        target_df.at[row_idx, "past_range_rep_sog"] = past_summary["range_rep_sog"]
        target_df.at[row_idx, "future_range_rep_sog"] = future_summary["range_rep_sog"]

        if pd.notna(past_summary["mean_rep_sog"]) and pd.notna(future_summary["mean_rep_sog"]):
            target_df.at[row_idx, "delta_mean_sog"] = (
                future_summary["mean_rep_sog"] - past_summary["mean_rep_sog"]
            )

        target_df.at[row_idx, "future_stop_ratio"] = future_summary["stop_ratio"]

        target_df.at[row_idx, "past_cog_valid_count"] = past_summary["n_cog_valid"]
        target_df.at[row_idx, "future_cog_valid_count"] = future_summary["n_cog_valid"]

        target_df.at[row_idx, "past_mean_rep_cog"] = past_summary["mean_rep_cog"]
        target_df.at[row_idx, "future_mean_rep_cog"] = future_summary["mean_rep_cog"]
        target_df.at[row_idx, "past_std_rep_cog"] = past_summary["std_rep_cog"]
        target_df.at[row_idx, "future_std_rep_cog"] = future_summary["std_rep_cog"]

        turn_change = circular_diff_scalar_deg(
            future_summary["mean_rep_cog"],
            past_summary["mean_rep_cog"]
        )

        target_df.at[row_idx, "turn_change_deg"] = turn_change
        target_df.at[row_idx, "turn_change_deg_abs"] = (
            abs(turn_change) if pd.notna(turn_change) else np.nan
        )

        target_df.at[row_idx, "speed_regime"] = get_speed_regime(
            past_summary["mean_rep_sog"]
        )
        target_df.at[row_idx, "accel_threshold_used"] = get_accel_threshold(
            past_summary["mean_rep_sog"]
        )
        target_df.at[row_idx, "decel_threshold_used"] = get_decel_threshold(
            past_summary["mean_rep_sog"]
        )

# ---------------------------------------------------------
# Assign final target class and target code
# ---------------------------------------------------------

target_df[TARGET_COL] = target_df.apply(assign_target, axis=1)

# Safety for newer Pandas / pyarrow
target_df["speed_regime"] = target_df["speed_regime"].astype("object")
target_df[TARGET_COL] = target_df[TARGET_COL].astype("object")

target_order = ["steady", "stop", "accelerate", "decelerate", "maneuver"]
target_map = {name: i for i, name in enumerate(target_order)}

target_df["target_code"] = target_df[TARGET_COL].map(target_map)

# Optional: nullable integer target code, keeps missing labels as <NA>
target_df["target_code"] = target_df["target_code"].astype("Int64")

moderate_steady_config = {
    "MODERATE_STEADY_MIN_PAST_SOG": MODERATE_STEADY_MIN_PAST_SOG,
    "MODERATE_STEADY_MIN_FUTURE_SOG": MODERATE_STEADY_MIN_FUTURE_SOG,
    "MODERATE_STEADY_MAX_ABS_DELTA_MEAN_SOG": MODERATE_STEADY_MAX_ABS_DELTA_MEAN_SOG,
    "MODERATE_STEADY_MAX_WINDOW_SOG_STD": MODERATE_STEADY_MAX_WINDOW_SOG_STD,
    "MODERATE_STEADY_MAX_TURN_CHANGE_DEG": MODERATE_STEADY_MAX_TURN_CHANGE_DEG,
    "MODERATE_STEADY_MAX_FUTURE_STOP_RATIO": MODERATE_STEADY_MAX_FUTURE_STOP_RATIO,
    "definition_note": "steady is positive stable-moving evidence, not residual fallback",
}

with open(FINAL_DIR / "moderate_steady_config.json", "w") as f:
    json.dump(moderate_steady_config, f, indent=2)

# ---------------------------------------------------------
# Persist target_df
# ---------------------------------------------------------

target_df.to_csv(FINAL_DIR / "target_df.csv", index=False)

print("target_df:", target_df.shape)

print("\nModerate steady configuration:")
print(pd.Series(moderate_steady_config))

print("\nTarget distribution:")
print(target_df[TARGET_COL].value_counts(dropna=False))

print("\nTarget distribution normalized:")
print(target_df[TARGET_COL].value_counts(normalize=True, dropna=False).round(4))

print("\nSpeed regime distribution:")
print(target_df["speed_regime"].value_counts(dropna=False))

print("\nSaved:", FINAL_DIR / "target_df.csv")

# Note: the maneuver→steady correction is evaluated in the training/evaluation notebook.
# This preprocessing notebook intentionally keeps the moderate steady label definition unchanged.


### Interpretation of target construction

The target dataframe keeps the full continuous timeline, so it contains the same **1,016,245** rows as `slice_level_df`. Most rows are unlabeled (**915,690**, or **90.11%**) because the notebook intentionally avoids forcing empty or ambiguous slices into a behavior class.

The class priority is `stop → maneuver → accelerate → decelerate → steady`. The `steady` class is assigned only when the future context shows continued movement and small speed/course changes. Therefore, the low number of `steady` rows in `target_df` is expected and reflects a conservative positive definition rather than a default catch-all label.

In [ ]:

# =========================================================
# Step 4: Build leakage-safe modelling dataframe and persist model_df
# Integrated version:
#   1) runs a zero-speed steady-label repair check in the prepared data,
#   2) creates rule-overlap diagnostic columns,
#   3) masks unreliable low-speed COG-derived model features,
#   4) adds past/current stability features to make the steady class less residual,
#   5) adds the final 48-feature inference-safe behavior trend scores,
#   6) persists model-ready data so training no longer has to patch labels/features.
# =========================================================

TARGET_CODE_COL = "target_code"

# Keep these repair / diagnostic constants aligned with the target-construction cell.
ZERO_SPEED_STEADY_REP_SOG_MAX = 0.10
ZERO_SPEED_STEADY_ROLL_MAX = 0.10
STRONG_MANEUVER_TURN_THRESHOLD_DEG = 60.0
EXTREME_MANEUVER_TURN_THRESHOLD_DEG = 90.0
STRONG_LAG_COG_JUMP_DEG = 90.0

# Moderate steady-label experiment: define positive past/current stability evidence.
# These are model features because they use only current/past slice values, not future target windows.
# They remain more conservative than the baseline candidate to avoid using steady as a broad fallback.
STEADY_MIN_MOVING_SOG = 1.0
STEADY_MAX_ABS_SOG_DELTA = 0.45
STEADY_MAX_ABS_COG_DELTA = 30.0
STEADY_MAX_ROLL3_STD = 0.40
STEADY_MAX_MOTION_STABILITY_SCORE = 1.35

REPAIR_ZERO_SPEED_STEADY_TO_STOP = True
ADD_RULE_OVERLAP_DIAGNOSTICS = True
MASK_LOW_SPEED_COG_FEATURES = True


def _num_col(df, col, default=np.nan):
    """Return a numeric Series aligned to df even when col is missing."""
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(default, index=df.index, dtype="float64")


def add_behavior_rule_flags(df):
    """
    Add rule-evidence columns used for diagnostics.

    Important:
    These columns are diagnostic/decision-support columns, not ordinary model
    features. They intentionally expose overlapping evidence such as
    ACCEL+MANEUVER or STOP+DECEL so confusion-matrix errors can be interpreted.
    """
    df = df.copy()

    rep_sog = _num_col(df, "rep_sog")
    roll2 = _num_col(df, "rep_sog_roll2")
    roll3 = _num_col(df, "rep_sog_roll3")
    future_stop_ratio = _num_col(df, "future_stop_ratio")

    stop_by_ratio = future_stop_ratio.notna() & (future_stop_ratio >= STOP_RATIO_THRESHOLD)
    stop_by_zero_speed = (
        rep_sog.notna() &
        (rep_sog <= STOP_SOG_THRESHOLD) &
        (roll2.fillna(rep_sog) <= STOP_SOG_THRESHOLD) &
        (roll3.fillna(rep_sog) <= STOP_SOG_THRESHOLD)
    )
    df["is_stop_rule"] = stop_by_ratio | stop_by_zero_speed

    accel_threshold = _num_col(df, "accel_threshold_used")
    decel_threshold = _num_col(df, "decel_threshold_used")
    delta_mean_sog = _num_col(df, "delta_mean_sog")

    d1 = _num_col(df, "rep_sog_delta_1")
    d2 = _num_col(df, "rep_sog_delta_2")
    fallback_delta = pd.concat([d1, d2], axis=1)

    # Prefer the same aggregate delta used by target construction. If absent,
    # fallback lag deltas provide diagnostic evidence only.
    effective_accel_delta = delta_mean_sog.fillna(fallback_delta.max(axis=1))
    effective_decel_delta = delta_mean_sog.fillna(fallback_delta.min(axis=1))

    df["is_accel_rule"] = (
        effective_accel_delta.notna() &
        accel_threshold.notna() &
        (effective_accel_delta >= accel_threshold)
    )
    df["is_decel_rule"] = (
        effective_decel_delta.notna() &
        decel_threshold.notna() &
        (effective_decel_delta <= -decel_threshold)
    )

    turn_change = _num_col(df, "turn_change_deg_abs")
    df["is_maneuver_rule"] = (
        turn_change.notna() &
        rep_sog.notna() &
        (rep_sog >= MIN_VALID_COG_SOG) &
        (turn_change >= MANEUVER_TURN_THRESHOLD)
    )

    lag1 = _num_col(df, "rep_cog_diff_lag1").abs()
    lag2 = _num_col(df, "rep_cog_diff_lag2").abs()
    df["strong_lag_cog_jump_flag"] = (
        (rep_sog >= MIN_VALID_COG_SOG) &
        ((lag1 >= STRONG_LAG_COG_JUMP_DEG) | (lag2 >= STRONG_LAG_COG_JUMP_DEG))
    )

    rule_cols = ["is_stop_rule", "is_accel_rule", "is_decel_rule", "is_maneuver_rule"]
    df["rule_overlap_count"] = df[rule_cols].sum(axis=1)

    def _signature(row):
        parts = []
        if row["is_stop_rule"]:
            parts.append("STOP")
        if row["is_accel_rule"]:
            parts.append("ACCEL")
        if row["is_decel_rule"]:
            parts.append("DECEL")
        if row["is_maneuver_rule"]:
            parts.append("MANEUVER")
        return "+".join(parts) if parts else "STEADY"

    df["rule_signature"] = df.apply(_signature, axis=1)

    df["maneuver_strength_bin"] = pd.cut(
        turn_change,
        bins=[
            -np.inf,
            MANEUVER_TURN_THRESHOLD,
            STRONG_MANEUVER_TURN_THRESHOLD_DEG,
            EXTREME_MANEUVER_TURN_THRESHOLD_DEG,
            np.inf,
        ],
        labels=["below_threshold", "borderline_45_60", "strong_60_90", "extreme_gt_90"],
    )

    df["cog_reliable"] = (rep_sog >= MIN_VALID_COG_SOG).astype(int)
    return df


def repair_zero_speed_steady_labels_in_model_df(df):
    """
    Repair the clearest target contradiction found during validation diagnostics:
    rows labeled as steady even though the current and rolling representative SOG
    values are essentially zero.

    This repair is applied in data preparation so train/validation/test y files
    and diagnostic dataframes remain aligned.
    """
    df = df.copy()

    steady_code = target_map.get("steady")
    stop_code = target_map.get("stop")

    if steady_code is None or stop_code is None:
        print("Zero-speed steady repair skipped: target_map does not contain steady/stop.")
        df["zero_speed_steady_repaired_to_stop"] = False
        return df, 0

    rep_sog = _num_col(df, "rep_sog")
    roll2 = _num_col(df, "rep_sog_roll2")
    roll3 = _num_col(df, "rep_sog_roll3")

    near_zero_speed = (
        (rep_sog <= ZERO_SPEED_STEADY_REP_SOG_MAX) &
        (roll2.fillna(rep_sog) <= ZERO_SPEED_STEADY_ROLL_MAX) &
        (roll3.fillna(rep_sog) <= ZERO_SPEED_STEADY_ROLL_MAX)
    )

    current_is_steady = df[TARGET_COL].astype(str).eq("steady")
    repair_mask = current_is_steady & near_zero_speed

    df["zero_speed_steady_repaired_to_stop"] = repair_mask

    n_repaired = int(repair_mask.sum())
    if n_repaired > 0:
        df.loc[repair_mask, TARGET_COL] = "stop"
        df.loc[repair_mask, TARGET_CODE_COL] = stop_code

    return df, n_repaired


def mask_low_speed_cog_features(df, feature_cols):
    """
    At low speed, COG can be unstable. Mask COG/turn/heading/course-derived
    model features when rep_sog < MIN_VALID_COG_SOG and keep a cog_reliable
    feature so the model can distinguish masked rows.
    """
    df = df.copy()

    if "cog_reliable" not in df.columns:
        df["cog_reliable"] = (_num_col(df, "rep_sog") >= MIN_VALID_COG_SOG).astype(int)

    if "cog_reliable" not in feature_cols:
        feature_cols = feature_cols + ["cog_reliable"]

    cog_feature_cols = [
        c for c in feature_cols
        if (
            c != "cog_reliable"
            and (
                "cog" in c.lower()
                or "turn" in c.lower()
                or "heading" in c.lower()
                or "course" in c.lower()
            )
        )
    ]

    low_speed_mask = df["cog_reliable"].astype(int).eq(0)
    if cog_feature_cols and low_speed_mask.any():
        df.loc[low_speed_mask, cog_feature_cols] = 0.0

    return df, feature_cols, cog_feature_cols, int(low_speed_mask.sum())


def add_past_current_stability_features(df):
    """
    Add positive steady-motion features using only current/past information.

    Motivation:
    Earlier pipeline versions risked making `steady` behave like a residual class.
    In this final version, `steady` is positively defined during target construction.
    These features give the model current/past-only evidence of stable movement.

    No future target-window columns are used here.
    """
    df = df.copy()

    d1 = _num_col(df, "rep_sog_delta_1")
    d2 = _num_col(df, "rep_sog_delta_2")
    c1 = _num_col(df, "rep_cog_diff_lag1")
    c2 = _num_col(df, "rep_cog_diff_lag2")
    rep_sog = _num_col(df, "rep_sog")

    df["abs_rep_sog_delta_1"] = d1.abs()
    df["abs_rep_sog_delta_2"] = d2.abs()
    df["abs_rep_cog_diff_lag1"] = c1.abs()
    df["abs_rep_cog_diff_lag2"] = c2.abs()

    # Lower score means more stable speed. NaNs are treated as 0 contribution
    # after the model's normal numeric conversion / estimator missing handling.
    df["sog_stability_score"] = (
        df["abs_rep_sog_delta_1"].fillna(0) +
        df["abs_rep_sog_delta_2"].fillna(0)
    )

    # Lower score means more stable course. Use absolute circular lagged changes.
    df["cog_stability_score"] = (
        df["abs_rep_cog_diff_lag1"].fillna(0) +
        df["abs_rep_cog_diff_lag2"].fillna(0)
    )

    # Combined stability score. The 0.02 factor keeps degrees from dominating knots.
    df["motion_stability_score"] = (
        df["sog_stability_score"] + 0.02 * df["cog_stability_score"]
    )

    # Rolling local speed variability: lower values suggest steady speed.
    df["rep_sog_roll3_std"] = (
        df.groupby(MMSI_COL)["rep_sog"]
          .transform(lambda x: x.rolling(window=3, min_periods=1).std())
    )

    # Diagnostic moving-steady candidate flag.
    # This column is persisted only for inspection/post-hoc diagnostics and is
    # NOT included in the final model feature matrix. The final paper model
    # instead uses continuous inference-safe trend features below.
    cog_reliable_series = (
        df["cog_reliable"].astype(int)
        if "cog_reliable" in df.columns
        else (rep_sog >= MIN_VALID_COG_SOG).astype(int)
    )
    strong_jump_series = (
        df["strong_lag_cog_jump_flag"].astype(int)
        if "strong_lag_cog_jump_flag" in df.columns
        else pd.Series(0, index=df.index, dtype="int64")
    )
    roll3_std = pd.to_numeric(df["rep_sog_roll3_std"], errors="coerce")
    motion_score = pd.to_numeric(df["motion_stability_score"], errors="coerce")

    df["is_moving_steady_candidate"] = (
        (rep_sog >= STEADY_MIN_MOVING_SOG) &
        (df["abs_rep_sog_delta_1"].fillna(0) <= STEADY_MAX_ABS_SOG_DELTA) &
        (df["abs_rep_sog_delta_2"].fillna(0) <= STEADY_MAX_ABS_SOG_DELTA) &
        (df["abs_rep_cog_diff_lag1"].fillna(0) <= STEADY_MAX_ABS_COG_DELTA) &
        (df["abs_rep_cog_diff_lag2"].fillna(0) <= STEADY_MAX_ABS_COG_DELTA) &
        (roll3_std.fillna(0) <= STEADY_MAX_ROLL3_STD) &
        (motion_score.fillna(np.inf) <= STEADY_MAX_MOTION_STABILITY_SCORE) &
        (strong_jump_series == 0) &
        (cog_reliable_series == 1)
    ).astype(int)

    # Final inference-safe behavior trend features used by the 48-feature model.
    # These replace the earlier binary moving-steady candidate as model input and
    # use only current/past values available at inference time.
    eps = 0.1
    rep_sog_roll3 = _num_col(df, "rep_sog_roll3")
    stop_tendency = (rep_sog_roll3 - rep_sog) / (rep_sog_roll3 + eps)
    df["stop_tendency_score"] = (
        stop_tendency
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .clip(-5.0, 5.0)
    )

    maneuver_intensity = c1.abs() / (1.0 + d1.abs())
    df["maneuver_intensity_score"] = (
        maneuver_intensity
        .where(cog_reliable_series.astype(int).eq(1), 0.0)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .clip(0.0, 180.0)
    )

    return df


model_df = target_df.copy()
model_df = model_df.sort_values([MMSI_COL, "slice_start"]).reset_index(drop=True)

model_df = model_df[
    model_df[TARGET_COL].notna() &
    model_df["has_data"] &
    model_df["enough_data_for_repr"]
].copy()

# Lag features per vessel
for lag in range(1, N_LAGS + 1):
    for col in ["rep_sog", "rep_cog", "n_msgs", "coverage_seconds", "cog_valid_for_maneuver"]:
        model_df[f"{col}_lag{lag}"] = model_df.groupby(MMSI_COL)[col].shift(lag)

# Past-only derived features
model_df["rep_sog_delta_1"] = model_df["rep_sog"] - model_df["rep_sog_lag1"]
model_df["rep_sog_delta_2"] = model_df["rep_sog_lag1"] - model_df["rep_sog_lag2"]

model_df["rep_cog_diff_lag1"] = circular_diff_series_deg(model_df["rep_cog"], model_df["rep_cog_lag1"])
model_df["rep_cog_diff_lag2"] = circular_diff_series_deg(model_df["rep_cog_lag1"], model_df["rep_cog_lag2"])

model_df["rep_sog_roll2"] = (
    model_df.groupby(MMSI_COL)["rep_sog"]
    .transform(lambda s: s.rolling(window=2, min_periods=1).mean())
)

model_df["rep_sog_roll3"] = (
    model_df.groupby(MMSI_COL)["rep_sog"]
    .transform(lambda s: s.rolling(window=3, min_periods=1).mean())
)

model_df["n_msgs_roll3"] = (
    model_df.groupby(MMSI_COL)["n_msgs"]
    .transform(lambda s: s.rolling(window=3, min_periods=1).mean())
)

model_df["coverage_roll3"] = (
    model_df.groupby(MMSI_COL)["coverage_seconds"]
    .transform(lambda s: s.rolling(window=3, min_periods=1).mean())
)

# Repair zero-speed steady labels after rolling features exist.
label_repair_summary = {"zero_speed_steady_to_stop": 0}
if REPAIR_ZERO_SPEED_STEADY_TO_STOP:
    model_df, n_repaired = repair_zero_speed_steady_labels_in_model_df(model_df)
    label_repair_summary["zero_speed_steady_to_stop"] = int(n_repaired)
else:
    model_df["zero_speed_steady_repaired_to_stop"] = False

# Add rule-overlap diagnostics after repair so target labels and diagnostics are persisted together.
if ADD_RULE_OVERLAP_DIAGNOSTICS:
    model_df = add_behavior_rule_flags(model_df)

# Add explicit past/current stability features to help the model learn steady movement
# as a positive pattern from inference-safe evidence.
model_df = add_past_current_stability_features(model_df)

# Time features: current slice only, no future leakage
model_df["hour"] = model_df["slice_start"].dt.hour
model_df["dayofweek"] = model_df["slice_start"].dt.dayofweek

model_df["hour_sin"] = np.sin(2 * np.pi * model_df["hour"] / 24.0)
model_df["hour_cos"] = np.cos(2 * np.pi * model_df["hour"] / 24.0)
model_df["dow_sin"] = np.sin(2 * np.pi * model_df["dayofweek"] / 7.0)
model_df["dow_cos"] = np.cos(2 * np.pi * model_df["dayofweek"] / 7.0)

feature_cols = [
    # base
    "rep_sog",
    "rep_cog",
    "sog_mean",
    "sog_std",
    "sog_min",
    "sog_max",
    "cog_circular_std",
    "n_msgs",
    "coverage_seconds",
    "cog_valid_for_maneuver",

    # lags
    "rep_sog_lag1", "rep_sog_lag2", "rep_sog_lag3",
    "rep_cog_lag1", "rep_cog_lag2", "rep_cog_lag3",
    "n_msgs_lag1", "n_msgs_lag2", "n_msgs_lag3",
    "coverage_seconds_lag1", "coverage_seconds_lag2", "coverage_seconds_lag3",
    "cog_valid_for_maneuver_lag1", "cog_valid_for_maneuver_lag2", "cog_valid_for_maneuver_lag3",

    # derived
    "rep_sog_delta_1",
    "rep_sog_delta_2",
    "rep_cog_diff_lag1",
    "rep_cog_diff_lag2",
    "rep_sog_roll2",
    "rep_sog_roll3",
    "n_msgs_roll3",
    "coverage_roll3",

    # steady/stability features: current/past only
    "abs_rep_sog_delta_1",
    "abs_rep_sog_delta_2",
    "abs_rep_cog_diff_lag1",
    "abs_rep_cog_diff_lag2",
    "sog_stability_score",
    "cog_stability_score",
    "motion_stability_score",
    "rep_sog_roll3_std",

    # inference-safe behavior trend features: current/past only
    "stop_tendency_score",
    "maneuver_intensity_score",

    # time
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
]

# Convert boolean-like columns for model compatibility.
bool_like_cols = [
    "cog_valid_for_maneuver",
    "cog_valid_for_maneuver_lag1",
    "cog_valid_for_maneuver_lag2",
    "cog_valid_for_maneuver_lag3",
    "strong_lag_cog_jump_flag",
    "zero_speed_steady_repaired_to_stop",
    "is_stop_rule",
    "is_accel_rule",
    "is_decel_rule",
    "is_maneuver_rule",
    "is_moving_steady_candidate",
]

for col in bool_like_cols:
    if col in model_df.columns:
        model_df[col] = model_df[col].fillna(False).astype("int64")

# Add and apply low-speed COG reliability feature/masking in prepared data.
masked_cog_feature_cols = []
low_speed_cog_masked_rows = 0
if MASK_LOW_SPEED_COG_FEATURES:
    model_df, feature_cols, masked_cog_feature_cols, low_speed_cog_masked_rows = mask_low_speed_cog_features(
        model_df,
        feature_cols,
    )

# Numeric conversion after all feature engineering/masking.
for col in feature_cols:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

# Final inference-safety checks for the paper/repository version.
assert "is_moving_steady_candidate" not in feature_cols, (
    "Diagnostic moving-steady candidate must not be part of the final model feature set."
)
for _required_col in ["stop_tendency_score", "maneuver_intensity_score", "cog_reliable"]:
    assert _required_col in feature_cols, f"Missing final model feature: {_required_col}"
_future_like_model_cols = [c for c in feature_cols if c.startswith("future_") or "future" in c.lower()]
assert not _future_like_model_cols, f"Future-window columns found in model features: {_future_like_model_cols}"
assert len(feature_cols) == 48, f"Expected 48 final model features, got {len(feature_cols)}"

model_df["target_code"] = model_df["target_code"].astype(int)
model_df[TARGET_COL] = model_df[TARGET_COL].astype("object")

model_df.to_csv(FINAL_DIR / "model_df.csv", index=False)

with open(FINAL_DIR / "feature_cols.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

with open(FINAL_DIR / "target_map.json", "w") as f:
    json.dump(target_map, f, indent=2)

preprocessing_update_summary = {
    "experiment": "MODERATE_STEADY_LABEL",
    "moderate_steady_config": moderate_steady_config if "moderate_steady_config" in globals() else {},
    "label_repair_summary": label_repair_summary,
    "mask_low_speed_cog_features": bool(MASK_LOW_SPEED_COG_FEATURES),
    "low_speed_cog_masked_rows": int(low_speed_cog_masked_rows),
    "masked_cog_feature_cols": masked_cog_feature_cols,
    "added_feature_cols": (
        (["cog_reliable"] if "cog_reliable" in feature_cols else []) +
        [
            "abs_rep_sog_delta_1",
            "abs_rep_sog_delta_2",
            "abs_rep_cog_diff_lag1",
            "abs_rep_cog_diff_lag2",
            "sog_stability_score",
            "cog_stability_score",
            "motion_stability_score",
            "rep_sog_roll3_std",
            "stop_tendency_score",
            "maneuver_intensity_score",
        ]
    ),
    "excluded_from_final_model_features": [
        "is_moving_steady_candidate",
    ],
    "diagnostic_columns_added": [
        "zero_speed_steady_repaired_to_stop",
        "is_stop_rule",
        "is_accel_rule",
        "is_decel_rule",
        "is_maneuver_rule",
        "rule_overlap_count",
        "rule_signature",
        "maneuver_strength_bin",
        "strong_lag_cog_jump_flag",
        "cog_reliable",
        "is_moving_steady_candidate",
    ],
}

with open(FINAL_DIR / "preprocessing_update_summary.json", "w") as f:
    json.dump(preprocessing_update_summary, f, indent=2)

print("model_df:", model_df.shape)
print("Feature count:", len(feature_cols))
print("\nLabel repair summary:")
print(pd.Series(label_repair_summary))
print("\nLow-speed COG masking:")
print("Rows masked:", low_speed_cog_masked_rows)
print("Feature columns masked:", masked_cog_feature_cols)
if "is_moving_steady_candidate" in model_df.columns:
    print("Diagnostic moving-steady candidate rows:", int(model_df["is_moving_steady_candidate"].sum()))
print("Inference-safe trend features included:", ["stop_tendency_score", "maneuver_intensity_score"])
print("Final model features include is_moving_steady_candidate?:", "is_moving_steady_candidate" in feature_cols)
print("\nClass distribution after preprocessing repairs:")
print(model_df[TARGET_COL].value_counts(normalize=True).round(4))
print("\nRule signature distribution:")
if "rule_signature" in model_df.columns:
    print(model_df["rule_signature"].value_counts().head(20))
print("\nSaved:", FINAL_DIR / "model_df.csv")


### Interpretation of model-ready dataframe

`model_df` is much smaller than `target_df` because it keeps only rows with a non-missing target label, current-slice data, and enough information for a representative current slice. In this final repository version, this produces **91,021** model-ready rows with **48** final model features.

The final 48-feature matrix excludes the earlier diagnostic `is_moving_steady_candidate` flag and includes two deployable current/past-only trend features:

- `stop_tendency_score`
- `maneuver_intensity_score`

The final class distribution is still highly imbalanced, dominated by `stop` (**84.36%**). This is expected in AIS behavior prediction because stopped or near-stopped vessel states are much more frequent than acceleration, deceleration, maneuver, or stable-moving cases.

Low-speed COG-derived features are masked for **76,940** rows because COG can be unreliable when vessels are stopped or moving very slowly. The separate `cog_reliable` feature lets the model distinguish reliable and masked course-related information.


In [ ]:
# =========================================================
# Step 4: Split and persist diagnosable train/valid/test files as CSV
# =========================================================

from sklearn.model_selection import GroupShuffleSplit, train_test_split

X = model_df[feature_cols].copy()
y = model_df["target_code"].astype(int).copy()

if SPLIT_BY_MMSI:
    groups = model_df[MMSI_COL].copy()

    gss1 = GroupShuffleSplit(n_splits=1, test_size=VALID_TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, temp_idx = next(gss1.split(X, y, groups=groups))

    train_df = model_df.iloc[train_idx].copy()
    temp_df = model_df.iloc[temp_idx].copy()

    X_temp = temp_df[feature_cols].copy()
    y_temp = temp_df["target_code"].astype(int).copy()
    groups_temp = temp_df[MMSI_COL].copy()

    gss2 = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE_WITHIN_TEMP, random_state=RANDOM_STATE)
    valid_rel_idx, test_rel_idx = next(gss2.split(X_temp, y_temp, groups=groups_temp))

    valid_df = temp_df.iloc[valid_rel_idx].copy()
    test_df = temp_df.iloc[test_rel_idx].copy()
    split_mode = "mmsi_held_out"

else:
    train_df, temp_df = train_test_split(
        model_df,
        test_size=VALID_TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=model_df["target_code"]
    )

    valid_df, test_df = train_test_split(
        temp_df,
        test_size=TEST_SIZE_WITHIN_TEMP,
        random_state=RANDOM_STATE,
        stratify=temp_df["target_code"]
    )
    split_mode = "row_level_stratified"

# Reset index so train_df/valid_df/test_df align exactly with X/y files.
train_df = train_df.sort_values([MMSI_COL, "slice_start"]).reset_index(drop=True)
valid_df = valid_df.sort_values([MMSI_COL, "slice_start"]).reset_index(drop=True)
test_df = test_df.sort_values([MMSI_COL, "slice_start"]).reset_index(drop=True)

# Persist complete rows for diagnostics.
train_df.to_csv(FINAL_DIR / "train_df.csv", index=False)
valid_df.to_csv(FINAL_DIR / "valid_df.csv", index=False)
test_df.to_csv(FINAL_DIR / "test_df.csv", index=False)

# Persist model matrices separately for fast training.
X_train = train_df[feature_cols].copy()
y_train = train_df["target_code"].astype(int).copy()

X_valid = valid_df[feature_cols].copy()
y_valid = valid_df["target_code"].astype(int).copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["target_code"].astype(int).copy()

X_train.to_csv(FINAL_DIR / "X_train.csv", index=False)
X_valid.to_csv(FINAL_DIR / "X_valid.csv", index=False)
X_test.to_csv(FINAL_DIR / "X_test.csv", index=False)

y_train.to_frame("target_code").to_csv(FINAL_DIR / "y_train.csv", index=False)
y_valid.to_frame("target_code").to_csv(FINAL_DIR / "y_valid.csv", index=False)
y_test.to_frame("target_code").to_csv(FINAL_DIR / "y_test.csv", index=False)

split_summary = {
    "split_mode": split_mode,
    "random_state": RANDOM_STATE,
    "valid_test_size": VALID_TEST_SIZE,
    "test_size_within_temp": TEST_SIZE_WITHIN_TEMP,
    "train_shape": list(X_train.shape),
    "valid_shape": list(X_valid.shape),
    "test_shape": list(X_test.shape),
    "train_mmsi_count": int(train_df[MMSI_COL].nunique()),
    "valid_mmsi_count": int(valid_df[MMSI_COL].nunique()),
    "test_mmsi_count": int(test_df[MMSI_COL].nunique()),
    "target_map": target_map,
    "feature_cols": feature_cols,
    "preprocessing_update_summary": preprocessing_update_summary if "preprocessing_update_summary" in globals() else {},
}

with open(FINAL_DIR / "split_summary.json", "w") as f:
    json.dump(split_summary, f, indent=2)

print("Split mode:", split_mode)
print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)
print("Test shape :", X_test.shape)

print("\nUnique MMSIs:")
print("Train:", train_df[MMSI_COL].nunique())
print("Valid:", valid_df[MMSI_COL].nunique())
print("Test :", test_df[MMSI_COL].nunique())

print("\nClass distribution:")
print("Train:\n", y_train.value_counts(normalize=True).sort_index().round(4))
print("Valid:\n", y_valid.value_counts(normalize=True).sort_index().round(4))
print("Test :\n", y_test.value_counts(normalize=True).sort_index().round(4))

print("\nSaved all reusable training/evaluation files to:", FINAL_DIR)

### Interpretation of train/validation/test split

The final split is **MMSI-held-out**, not row-level random. This is important because it evaluates whether the model generalizes to unseen vessels rather than memorizing vessel-specific patterns.

The validation and test distributions are not perfectly identical to the training distribution because group-based splitting preserves MMSI separation rather than exact class stratification. This is acceptable and more realistic for this project.

In [ ]:
# =========================================================
# Optional Step: Create balanced training dataset by downsampling train only
# Validation and test are intentionally kept unchanged.
# =========================================================
import shutil

SEED = 42
SOURCE_DIR = FINAL_DIR
BALANCED_DIR = PREP_DIR / "balanced_train"
BALANCED_DIR.mkdir(parents=True, exist_ok=True)

# Load persisted train / valid / test data
X_train = pd.read_csv(SOURCE_DIR / "X_train.csv")
X_valid = pd.read_csv(SOURCE_DIR / "X_valid.csv")
X_test = pd.read_csv(SOURCE_DIR / "X_test.csv")

y_train = pd.read_csv(SOURCE_DIR / "y_train.csv")["target_code"].astype(int)
y_valid = pd.read_csv(SOURCE_DIR / "y_valid.csv")["target_code"].astype(int)
y_test = pd.read_csv(SOURCE_DIR / "y_test.csv")["target_code"].astype(int)

train_df = pd.read_csv(SOURCE_DIR / "train_df.csv")
valid_df = pd.read_csv(SOURCE_DIR / "valid_df.csv")
test_df = pd.read_csv(SOURCE_DIR / "test_df.csv")

with open(SOURCE_DIR / "target_map.json", "r") as f:
    target_map = json.load(f)

with open(SOURCE_DIR / "feature_cols.json", "r") as f:
    feature_cols = json.load(f)

print("Original train shape:", X_train.shape)
print("\nOriginal train target distribution:")
print(y_train.value_counts().sort_index())

# Build balanced training row index from train_df so diagnostics stay aligned.
class_counts = train_df["target_code"].value_counts().sort_index()
min_class_count = class_counts.min()

print("\nClass counts:")
print(class_counts)
print("\nDownsampling every class to:", min_class_count)

balanced_parts = []
for class_id, group in train_df.groupby("target_code"):
    sampled_group = group.sample(
        n=min_class_count,
        random_state=SEED,
        replace=False
    )
    balanced_parts.append(sampled_group)

train_df_balanced = (
    pd.concat(balanced_parts, axis=0)
    .sample(frac=1.0, random_state=SEED)
    .reset_index(drop=True)
)

X_train_balanced = train_df_balanced[feature_cols].copy()
y_train_balanced = train_df_balanced["target_code"].astype(int).copy()

print("\nBalanced train shape:", X_train_balanced.shape)
print("\nBalanced train target distribution:")
print(y_train_balanced.value_counts().sort_index())
print("\nBalanced train target distribution normalized:")
print(y_train_balanced.value_counts(normalize=True).sort_index().round(4))

# Save balanced training data + unchanged validation/test.
train_df_balanced.to_csv(BALANCED_DIR / "train_df.csv", index=False)
valid_df.to_csv(BALANCED_DIR / "valid_df.csv", index=False)
test_df.to_csv(BALANCED_DIR / "test_df.csv", index=False)

X_train_balanced.to_csv(BALANCED_DIR / "X_train.csv", index=False)
y_train_balanced.to_frame("target_code").to_csv(BALANCED_DIR / "y_train.csv", index=False)

X_valid.to_csv(BALANCED_DIR / "X_valid.csv", index=False)
X_test.to_csv(BALANCED_DIR / "X_test.csv", index=False)
y_valid.to_frame("target_code").to_csv(BALANCED_DIR / "y_valid.csv", index=False)
y_test.to_frame("target_code").to_csv(BALANCED_DIR / "y_test.csv", index=False)

with open(BALANCED_DIR / "target_map.json", "w") as f:
    json.dump(target_map, f, indent=2)

with open(BALANCED_DIR / "feature_cols.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

# Preserve metadata files needed by the training/evaluation notebook.
metadata_files_to_copy = [
    "split_summary.json",
    "moderate_steady_config.json",
    "preprocessing_update_summary.json",
]

for fname in metadata_files_to_copy:
    src = SOURCE_DIR / fname
    dst = BALANCED_DIR / fname
    if src.exists():
        shutil.copy2(src, dst)

# Preserve preprocessing update metadata for the training notebook.
preprocessing_update_summary_path = SOURCE_DIR / "preprocessing_update_summary.json"
if preprocessing_update_summary_path.exists():
    with open(preprocessing_update_summary_path, "r") as f:
        preprocessing_update_summary = json.load(f)
    with open(BALANCED_DIR / "preprocessing_update_summary.json", "w") as f:
        json.dump(preprocessing_update_summary, f, indent=2)
else:
    preprocessing_update_summary = {}


balanced_summary = {
    "source_dir": str(SOURCE_DIR),
    "balanced_dir": str(BALANCED_DIR),
    "strategy": "downsample_majority_classes_train_only",
    "random_seed": SEED,
    "min_class_count": int(min_class_count),
    "original_train_shape": list(X_train.shape),
    "balanced_train_shape": list(X_train_balanced.shape),
    "original_train_class_counts": {str(int(k)): int(v) for k, v in class_counts.items()},
    "balanced_train_class_counts": {
        str(int(k)): int(v)
        for k, v in y_train_balanced.value_counts().sort_index().items()
    },
    "target_map": target_map,
    "feature_cols": feature_cols,
    "preprocessing_update_summary": preprocessing_update_summary,
}

with open(BALANCED_DIR / "balanced_summary.json", "w") as f:
    json.dump(balanced_summary, f, indent=2)

print("\nSaved balanced dataset to:", BALANCED_DIR)
for p in sorted(BALANCED_DIR.glob("*")):
    print("-", p)

### Interpretation of balanced training export

The balanced dataset is created by downsampling the **training set only** to **1,325 rows per class**. Validation and test are intentionally left unchanged so evaluation still reflects the natural DS2 class imbalance.

This balanced export is useful for model experiments that need stronger minority-class learning, but final metrics should always be interpreted on the original validation/test distributions.

In [ ]:
# =========================================================
# Fast path for loading prepared AIS DS2 data
# =========================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

FINAL_DIR = Path("./ais_ds2_prepared/final")
# Or use the balanced-train version:
# FINAL_DIR = Path("./ais_ds2_prepared/balanced_train")

with open(FINAL_DIR / "feature_cols.json") as f:
    feature_cols = json.load(f)

with open(FINAL_DIR / "target_map.json") as f:
    target_map = json.load(f)

class_name_map = {int(v): k for k, v in target_map.items()}
CLASS_IDS = sorted(class_name_map.keys())
CLASS_NAMES = [class_name_map[i] for i in CLASS_IDS]
id_to_label = class_name_map
label_to_id = {v: k for k, v in id_to_label.items()}

X_train = pd.read_csv(FINAL_DIR / "X_train.csv")
X_valid = pd.read_csv(FINAL_DIR / "X_valid.csv")
X_test = pd.read_csv(FINAL_DIR / "X_test.csv")

y_train = pd.read_csv(FINAL_DIR / "y_train.csv")["target_code"].astype(int)
y_valid = pd.read_csv(FINAL_DIR / "y_valid.csv")["target_code"].astype(int)
y_test = pd.read_csv(FINAL_DIR / "y_test.csv")["target_code"].astype(int)

train_df = pd.read_csv(FINAL_DIR / "train_df.csv", parse_dates=["slice_start", "slice_end"], low_memory=False)
valid_df = pd.read_csv(FINAL_DIR / "valid_df.csv", parse_dates=["slice_start", "slice_end"], low_memory=False)
test_df = pd.read_csv(FINAL_DIR / "test_df.csv", parse_dates=["slice_start", "slice_end"], low_memory=False)

print("Loaded persisted train/valid/test data.")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)
print("Classes:", class_name_map)
print("Diagnostic valid_df columns available:", valid_df.columns.tolist()[:20], "...")

### Reuse note

This cell verifies that the persisted files can be loaded directly by the training/evaluation notebook. The loaded classes match the intended target map: `{0: 'steady', 1: 'stop', 2: 'accelerate', 3: 'decelerate', 4: 'maneuver'}`.

For the final paper/repository version, the persisted feature matrix should contain **48** model features, including `stop_tendency_score` and `maneuver_intensity_score`, and excluding `is_moving_steady_candidate`.


In [ ]:
# =========================================================
# Reusable diagnostics after training a model
# Requires: model, X_eval, y_eval, df_eval, id_to_label
# Example: model = best_model; X_eval = X_valid; y_eval = y_valid; df_eval = valid_df
# =========================================================

import matplotlib.pyplot as plt


def build_inspect_df(model, X_eval, y_eval, df_eval, id_to_label):
    y_pred = model.predict(X_eval)

    y_true_name = pd.Series(y_eval).map(id_to_label)
    y_pred_name = pd.Series(y_pred).map(id_to_label)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_eval)
        pred_confidence = proba.max(axis=1)
    else:
        pred_confidence = np.nan

    inspect_df = df_eval.reset_index(drop=True).copy()
    inspect_df["actual_class"] = y_true_name.reset_index(drop=True)
    inspect_df["predicted_class"] = y_pred_name.reset_index(drop=True)
    inspect_df["is_correct"] = inspect_df["actual_class"] == inspect_df["predicted_class"]
    inspect_df["pred_confidence"] = pred_confidence
    inspect_df["eval_row_id"] = np.arange(len(inspect_df))

    return inspect_df


def get_confusion_cases(inspect_df, actual_class, predicted_classes, n=20, sort_by_confidence=True, random_sample=False, seed=42):
    if isinstance(predicted_classes, str):
        predicted_classes = [predicted_classes]

    cases = inspect_df[
        (inspect_df["actual_class"] == actual_class) &
        (inspect_df["predicted_class"].isin(predicted_classes))
    ].copy()

    if cases.empty:
        print(f"No cases found: actual={actual_class}, predicted={predicted_classes}")
        return cases

    if random_sample:
        cases = cases.sample(min(n, len(cases)), random_state=seed)
    elif sort_by_confidence and "pred_confidence" in cases.columns:
        cases = cases.sort_values("pred_confidence", ascending=False).head(n)
    else:
        cases = cases.head(n)

    print(f"Showing {len(cases)} cases for actual={actual_class}, predicted={predicted_classes}")
    return cases


def inspect_time_neighborhood(selected_row, df_full, mmsi_col="MMSI", time_col="slice_start", minutes_before=30, minutes_after=30):
    df_full = df_full.copy()
    df_full[time_col] = pd.to_datetime(df_full[time_col])

    selected_mmsi = selected_row[mmsi_col]
    selected_time = pd.to_datetime(selected_row[time_col])

    start_time = selected_time - pd.Timedelta(minutes=minutes_before)
    end_time = selected_time + pd.Timedelta(minutes=minutes_after)

    neighborhood = df_full[
        (df_full[mmsi_col] == selected_mmsi) &
        (df_full[time_col] >= start_time) &
        (df_full[time_col] <= end_time)
    ].copy()

    neighborhood = neighborhood.sort_values(time_col)

    print(f"MMSI: {selected_mmsi}")
    print(f"Selected time: {selected_time}")
    print(f"Window: {start_time} to {end_time}")
    print(f"Rows found: {len(neighborhood)}")

    return neighborhood


def plot_error_neighborhood(neighborhood, selected_row=None, time_col="slice_start", sog_col="rep_sog", cog_col="rep_cog"):
    neighborhood = neighborhood.copy()
    neighborhood[time_col] = pd.to_datetime(neighborhood[time_col])
    neighborhood = neighborhood.sort_values(time_col)

    selected_time = None
    if selected_row is not None and time_col in selected_row:
        selected_time = pd.to_datetime(selected_row[time_col])

    plt.figure(figsize=(12, 4))
    plt.plot(neighborhood[time_col], neighborhood[sog_col], marker="o")
    if selected_time is not None:
        plt.axvline(selected_time, linestyle="--", label="selected sample")
    plt.title("SOG around selected error sample")
    plt.xlabel("Time")
    plt.ylabel(sog_col)
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(neighborhood[time_col], neighborhood[cog_col], marker="o")
    if selected_time is not None:
        plt.axvline(selected_time, linestyle="--", label="selected sample")
    plt.title("COG around selected error sample")
    plt.xlabel("Time")
    plt.ylabel(cog_col)
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


# Example after you train/select best_model:
# inspect_df = build_inspect_df(best_model, X_valid, y_valid, valid_df, id_to_label)
# maneuver_to_speed_change = get_confusion_cases(inspect_df, "maneuver", ["accelerate", "decelerate"], n=20)
# steady_to_other = get_confusion_cases(inspect_df, "steady", ["stop", "accelerate", "decelerate"], n=20)
# decelerate_to_maneuver = get_confusion_cases(inspect_df, "decelerate", "maneuver", n=20)
#
# selected = maneuver_to_speed_change.iloc[0]
# neighborhood = inspect_time_neighborhood(selected, valid_df, minutes_before=30, minutes_after=30)
# display(neighborhood)
# plot_error_neighborhood(neighborhood, selected)